# 03b (Kaggle GPU) — Eksperimen NRP 0 (`320` & `960`) untuk Mask R-CNN & Mask2Former
**Setting Kaggle:** Accelerator `GPU T4 x2` atau `P100`, Internet `ON`.

Notebook ini melengkapi variabel eksperimen **NRP akhiran 0 (`320` / `640` / `960`)** untuk **Model 2 (Mask R-CNN)** dan **Model 3 (Mask2Former)** (karena `640` sudah selesai sebelumnya di `kaggle_results_cp3.zip`).
- Menggunakan `getpass` untuk API Key Roboflow agar **API Key tidak pernah tersimpan di file `.ipynb` / GitHub**.
- Menggunakan `batch=1, GRAD_ACCUM=8` pada Mask2Former `960px` agar 100% aman dari *Out of Memory* di T4 15GB.


In [ ]:
!pip install -q roboflow pycocotools transformers accelerate timm pandas

import os, time, json, zipfile, getpass
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
from roboflow import Roboflow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

# API Key diminta saat runtime (AMAN untuk GitHub — tidak tersimpan di kode!)
ROBOFLOW_API_KEY = getpass.getpass('Masukkan Roboflow Private API Key: ')

CUSTOM_WORKSPACE = 'cdaaptnia'
CUSTOM_PROJECT   = 'test-hp-ruangan'
CUSTOM_VERSION   = 2

PROJECT_DIR = '/kaggle/working/cpmk3_320_960'
MAIN_COCO   = os.path.join(PROJECT_DIR, 'datasets', 'main_coco')
CUSTOM_COCO = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_coco')
LOGS_DIR    = os.path.join(PROJECT_DIR, 'logs')
RUNS_DIR    = os.path.join(PROJECT_DIR, 'runs')
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
if not os.path.exists(MAIN_COCO):
    rf.workspace('panopticindoor').project('panoptic-indoor-segmentation').version(5).download(
        'coco-segmentation', location=MAIN_COCO, overwrite=True
    )
if not os.path.exists(CUSTOM_COCO):
    rf.workspace(CUSTOM_WORKSPACE).project(CUSTOM_PROJECT).version(CUSTOM_VERSION).download(
        'coco-segmentation', location=CUSTOM_COCO, overwrite=True
    )

def find_coco_split(root_dir, split_candidates):
    for sp in split_candidates:
        sp_dir = os.path.join(root_dir, sp)
        ann = os.path.join(sp_dir, '_annotations.coco.json')
        if os.path.exists(ann):
            return sp_dir, ann
    raise FileNotFoundError(f'Tidak menemukan _annotations.coco.json di {root_dir}')

TRAIN_DIR, TRAIN_ANN = find_coco_split(MAIN_COCO, ['train'])
TEST_DIR,  TEST_ANN  = find_coco_split(MAIN_COCO, ['test', 'valid'])
HP_DIR,    HP_ANN    = find_coco_split(CUSTOM_COCO, ['test', 'valid', 'train'])

coco_tr = COCO(TRAIN_ANN)
used_ids = set(a['category_id'] for a in coco_tr.dataset['annotations'])
VALID_CATS = sorted([c for c in coco_tr.loadCats(coco_tr.getCatIds()) if c['id'] in used_ids], key=lambda x: x['id'])
CAT2CONTIG = {c['id']: i + 1 for i, c in enumerate(VALID_CATS)}
ID2NAME    = {i + 1: c['name'] for i, c in enumerate(VALID_CATS)}
NUM_CLASSES = len(VALID_CATS) + 1

M2F_ID2LABEL   = {i: c['name'] for i, c in enumerate(VALID_CATS)}
M2F_LABEL2ID   = {c['name']: i for i, c in enumerate(VALID_CATS)}
M2F_CAT2CONTIG = {c['id']: i for i, c in enumerate(VALID_CATS)}

def append_gpu_log(row_dict):
    p = os.path.join(LOGS_DIR, 'gpu_time_log_320_960.csv')
    df_new = pd.DataFrame([row_dict])
    df_all = pd.concat([pd.read_csv(p), df_new], ignore_index=True) if os.path.exists(p) else df_new
    df_all.to_csv(p, index=False)

print(f'Dataset siap! Jumlah kelas aktif: {len(VALID_CATS)}')


## 1. Mask R-CNN (`320` dan `960`)


In [ ]:
class CocoInstanceDataset(Dataset):
    def __init__(self, img_dir, ann_file, cat2contig, img_size=640):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.cat2contig = cat2contig
        self.img_size = img_size
        self.img_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        orig_w, orig_h = img.size
        img_tensor = torchvision.transforms.functional.to_tensor(
            img.resize((self.img_size, self.img_size), Image.BILINEAR)
        )
        sx, sy = self.img_size / orig_w, self.img_size / orig_h
        boxes, labels, masks = [], [], []
        for ann in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id)):
            if ann['category_id'] not in self.cat2contig:
                continue
            x, y, w, h = ann['bbox']
            if w <= 1 or h <= 1:
                continue
            m = self.coco.annToMask(ann)
            m_res = np.array(Image.fromarray(m).resize((self.img_size, self.img_size), Image.NEAREST), dtype=np.uint8)
            if m_res.sum() == 0:
                continue
            boxes.append([x * sx, y * sy, (x + w) * sx, (y + h) * sy])
            labels.append(self.cat2contig[ann['category_id']])
            masks.append(m_res)
        if not boxes:
            return img_tensor, {'boxes': torch.zeros((0, 4)), 'labels': torch.zeros((0,), dtype=torch.int64),
                                'masks': torch.zeros((0, self.img_size, self.img_size), dtype=torch.uint8)}
        return img_tensor, {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'masks': torch.tensor(np.stack(masks, axis=0), dtype=torch.uint8),
        }


def build_maskrcnn(num_classes, img_size):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT, min_size=img_size, max_size=img_size
    )
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes)
    return model


@torch.no_grad()
def eval_maskrcnn(model, img_dir, ann_file, img_size, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    name_to_id = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}
    res_b, res_s = [], []
    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        ow, oh = img.size
        t = torchvision.transforms.functional.to_tensor(img.resize((img_size, img_size), Image.BILINEAR)).to(device)
        out = model([t])[0]
        sx, sy = ow / img_size, oh / img_size
        for s, l, b, m in zip(out['scores'].cpu().numpy(), out['labels'].cpu().numpy(),
                              out['boxes'].cpu().numpy(), out['masks'].cpu().numpy()):
            if s < score_thresh:
                continue
            cname = ID2NAME.get(int(l))
            if cname not in name_to_id:
                continue
            cid = name_to_id[cname]
            x1, y1, x2, y2 = b[0] * sx, b[1] * sy, b[2] * sx, b[3] * sy
            res_b.append({'image_id': int(img_id), 'category_id': int(cid),
                          'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], 'score': float(s)})
            bm = np.array(Image.fromarray((m[0] >= 0.5).astype(np.uint8)).resize((ow, oh), Image.NEAREST), dtype=np.uint8)
            rle = mask_utils.encode(np.asfortranarray(bm))
            rle['counts'] = rle['counts'].decode('utf-8')
            res_s.append({'image_id': int(img_id), 'category_id': int(cid), 'segmentation': rle, 'score': float(s)})
    met = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if res_b:
        ev = COCOeval(coco_gt, coco_gt.loadRes(res_b), 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        met['box_mAP50_95'], met['box_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    if res_s:
        ev = COCOeval(coco_gt, coco_gt.loadRes(res_s), 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        met['mask_mAP50_95'], met['mask_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    return met


MR_EPOCHS = 15
MR_CFG = {320: 4, 960: 2}
mr_rows = []

for imgsz, bsz in MR_CFG.items():
    run_dir = os.path.join(RUNS_DIR, 'maskrcnn', f'maskrcnn_img{imgsz}')
    os.makedirs(run_dir, exist_ok=True)
    ckpt = os.path.join(run_dir, 'best.pth')
    csv_p = os.path.join(run_dir, 'results.csv')

    train_loader = DataLoader(CocoInstanceDataset(TRAIN_DIR, TRAIN_ANN, CAT2CONTIG, imgsz),
                              batch_size=bsz, shuffle=True, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))
    model = build_maskrcnn(NUM_CLASSES, imgsz).to(device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    t0, hist, best_loss = time.time(), [], float('inf')
    for ep in range(1, MR_EPOCHS + 1):
        model.train()
        ep_loss, n_st, ep_t0 = 0.0, 0, time.time()
        for imgs, tgs in train_loader:
            pairs = [(im.to(device), {k: v.to(device) for k, v in tg.items()}) for im, tg in zip(imgs, tgs) if len(tg['boxes']) > 0]
            if not pairs:
                continue
            im_l, tg_l = zip(*pairs)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                losses = sum(model(list(im_l), list(tg_l)).values())
            scaler.scale(losses).backward()
            scaler.step(opt)
            scaler.update()
            ep_loss += float(losses.item()); n_st += 1
        avg_l = ep_loss / max(n_st, 1)
        ep_s = round(time.time() - ep_t0, 1)
        print(f'[Mask R-CNN img={imgsz}] Epoch {ep}/{MR_EPOCHS} | loss={avg_l:.4f} | {ep_s}s')
        hist.append({'epoch': ep, 'train_loss': round(avg_l, 4), 'epoch_sec': ep_s})
        if avg_l < best_loss:
            best_loss = avg_l
            torch.save(model.state_dict(), ckpt)
    pd.DataFrame(hist).to_csv(csv_p, index=False)
    append_gpu_log({
        'model': 'MaskRCNN-R50-FPNv2', 'architecture_family': 'Two-Stage CNN',
        'imgsz': imgsz, 'epochs': MR_EPOCHS, 'batch_size': bsz,
        'gpu_name': torch.cuda.get_device_name(0), 'train_time_minutes': round((time.time() - t0) / 60.0, 2),
        'results_csv': csv_p,
    })

    model.load_state_dict(torch.load(ckpt, map_location=device))
    for lbl, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
        t_e = time.time()
        m = eval_maskrcnn(model, d_dir, d_ann, imgsz)
        m.update({'model': 'MaskRCNN-R50-FPNv2', 'architecture_family': 'Two-Stage CNN',
                  'imgsz': imgsz, 'dataset': lbl, 'eval_time_sec': round(time.time() - t_e, 2)})
        mr_rows.append(m)
    del model; torch.cuda.empty_cache()

df_mr_320_960 = pd.DataFrame(mr_rows)
df_mr_320_960.to_csv(os.path.join(LOGS_DIR, 'eval_maskrcnn_320_960.csv'), index=False)
df_mr_320_960


## 2. Mask2Former (`320` dan `960`)
Menggunakan `batch=2, GRAD_ACCUM=4` pada `320px` dan `batch=1, GRAD_ACCUM=8` pada `960px` (batch efektif sama-sama 8, aman dari OOM di T4 15GB).


In [ ]:
M2F_MODEL_ID = 'facebook/mask2former-swin-tiny-coco-instance'
M2F_EPOCHS = 15
M2F_CFG = {
    320: {'batch': 2, 'accum': 4},
    960: {'batch': 1, 'accum': 8},
}


class Mask2FormerCocoDataset(Dataset):
    def __init__(self, img_dir, ann_file, cat2contig, processor, img_size):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.cat2contig = cat2contig
        self.processor = processor
        self.img_size = img_size
        self.img_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        masks_list, classes_list = [], []
        for ann in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id)):
            if ann['category_id'] not in self.cat2contig:
                continue
            m = self.coco.annToMask(ann)
            if m.sum() == 0:
                continue
            m_res = np.array(Image.fromarray(m.astype(np.uint8)).resize((self.img_size, self.img_size), Image.NEAREST), dtype=np.float32)
            if m_res.sum() == 0:
                continue
            masks_list.append(m_res)
            classes_list.append(self.cat2contig[ann['category_id']])
        if not masks_list:
            return None
        inp = self.processor(images=img, return_tensors='pt')
        return {
            'pixel_values': inp['pixel_values'].squeeze(0),
            'pixel_mask': inp['pixel_mask'].squeeze(0),
            'mask_labels': torch.tensor(np.stack(masks_list, axis=0), dtype=torch.float32),
            'class_labels': torch.tensor(classes_list, dtype=torch.int64),
        }


def m2f_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'pixel_mask': torch.stack([b['pixel_mask'] for b in batch]),
        'mask_labels': [b['mask_labels'] for b in batch],
        'class_labels': [b['class_labels'] for b in batch],
    }


@torch.no_grad()
def eval_m2f(model, processor, img_dir, ann_file, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    name_to_id = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}
    res_b, res_s = [], []
    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        ow, oh = img.size
        inp = processor(images=img, return_tensors='pt').to(device)
        out = model(**inp)
        pred = processor.post_process_instance_segmentation(out, threshold=score_thresh, target_sizes=[(oh, ow)])[0]
        seg_map = pred['segmentation'].cpu().numpy()
        for si in pred['segments_info']:
            cname = M2F_ID2LABEL.get(int(si['label_id']))
            if cname not in name_to_id:
                continue
            cid = name_to_id[cname]
            bm = (seg_map == si['id']).astype(np.uint8)
            if bm.sum() == 0:
                continue
            ys, xs = np.where(bm == 1)
            x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
            sc = float(si['score'])
            res_b.append({'image_id': int(img_id), 'category_id': int(cid),
                          'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': sc})
            rle = mask_utils.encode(np.asfortranarray(bm))
            rle['counts'] = rle['counts'].decode('utf-8')
            res_s.append({'image_id': int(img_id), 'category_id': int(cid), 'segmentation': rle, 'score': sc})
    met = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if res_b:
        ev = COCOeval(coco_gt, coco_gt.loadRes(res_b), 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        met['box_mAP50_95'], met['box_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    if res_s:
        ev = COCOeval(coco_gt, coco_gt.loadRes(res_s), 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        met['mask_mAP50_95'], met['mask_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    return met


m2f_rows = []
for imgsz, cfg in M2F_CFG.items():
    bsz, acc = cfg['batch'], cfg['accum']
    run_dir = os.path.join(RUNS_DIR, 'mask2former', f'm2f_swin_tiny_img{imgsz}')
    os.makedirs(run_dir, exist_ok=True)
    ckpt_dir = os.path.join(run_dir, 'best_model')
    csv_p = os.path.join(run_dir, 'results.csv')

    proc = AutoImageProcessor.from_pretrained(M2F_MODEL_ID, size={'height': imgsz, 'width': imgsz})
    model = Mask2FormerForUniversalSegmentation.from_pretrained(
        M2F_MODEL_ID, id2label=M2F_ID2LABEL, label2id=M2F_LABEL2ID, ignore_mismatched_sizes=True
    ).to(device)

    loader = DataLoader(Mask2FormerCocoDataset(TRAIN_DIR, TRAIN_ANN, M2F_CAT2CONTIG, proc, imgsz),
                        batch_size=bsz, shuffle=True, num_workers=2, collate_fn=m2f_collate)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    t0, hist, best_loss = time.time(), [], float('inf')
    for ep in range(1, M2F_EPOCHS + 1):
        model.train()
        ep_loss, n_st, ep_t0 = 0.0, 0, time.time()
        opt.zero_grad()
        for st, batch in enumerate(loader, 1):
            if batch is None:
                continue
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                out = model(
                    pixel_values=batch['pixel_values'].to(device),
                    pixel_mask=batch['pixel_mask'].to(device),
                    mask_labels=[m.to(device) for m in batch['mask_labels']],
                    class_labels=[c.to(device) for c in batch['class_labels']],
                )
                loss = out.loss / acc
            scaler.scale(loss).backward()
            if st % acc == 0 or st == len(loader):
                scaler.step(opt); scaler.update(); opt.zero_grad()
            ep_loss += float(out.loss.item()); n_st += 1
        avg_l = ep_loss / max(n_st, 1)
        ep_s = round(time.time() - ep_t0, 1)
        print(f'[Mask2Former img={imgsz}] Epoch {ep}/{M2F_EPOCHS} | loss={avg_l:.4f} | {ep_s}s')
        hist.append({'epoch': ep, 'train_loss': round(avg_l, 4), 'epoch_sec': ep_s})
        if avg_l < best_loss:
            best_loss = avg_l
            model.save_pretrained(ckpt_dir)

    pd.DataFrame(hist).to_csv(csv_p, index=False)
    append_gpu_log({
        'model': 'Mask2Former-Swin-Tiny', 'architecture_family': 'Transformer',
        'imgsz': imgsz, 'epochs': M2F_EPOCHS, 'batch_size': bsz * acc,
        'gpu_name': torch.cuda.get_device_name(0), 'train_time_minutes': round((time.time() - t0) / 60.0, 2),
        'results_csv': csv_p,
    })

    model = Mask2FormerForUniversalSegmentation.from_pretrained(ckpt_dir).to(device)
    for lbl, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
        t_e = time.time()
        m = eval_m2f(model, proc, d_dir, d_ann)
        m.update({'model': 'Mask2Former-Swin-Tiny', 'architecture_family': 'Transformer',
                  'imgsz': imgsz, 'dataset': lbl, 'eval_time_sec': round(time.time() - t_e, 2)})
        m2f_rows.append(m)
    del model; torch.cuda.empty_cache()

df_m2f_320_960 = pd.DataFrame(m2f_rows)
df_m2f_320_960.to_csv(os.path.join(LOGS_DIR, 'eval_mask2former_320_960.csv'), index=False)
df_m2f_320_960


## 3. Paketkan Semua Hasil (`320` & `960`) ke ZIP


In [ ]:
from IPython.display import FileLink

zip_path = '/kaggle/working/kaggle_results_320_960.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for root, _, files in os.walk(PROJECT_DIR):
        for fn in files:
            if fn.endswith('.csv'):
                full = os.path.join(root, fn)
                zf.write(full, arcname=os.path.relpath(full, PROJECT_DIR))
print('Siap download:', zip_path)
FileLink('kaggle_results_320_960.zip')
